## Chunking strategies

> How data should be indexed depends on how you intend to retrieve it later

### Why the need to chunk?
- **large documents** won’t fit into the context window of the generative model (or embedding model used to ingest the data)
- **retrieval performance** can degrade if chunks are too large or too small
- **cost considerations**: larger chunks may lead to higher costs during embedding and retrieval

### Chunking considerations
- **overlapping chunks** are useful when you want to ensure that important context is not lost between chunks
- **chunk size matter** because it affects retrieval performance and cost
    - smaller chunks may lead to more precise retrieval but can increase the number of chunks to process and can lose context
    - larger chunks may retain more context but can be less precise and more costly to process
- **preserving structure** (like headers in documents) can help maintain context and improve retrieval




## Text splitters

**Text splitters** break large docs into smaller chunks that will be retrievable individually and fit within model context window limit.

There are several strategies for splitting documents, each with its own advantages.

- Text structure-based splitters (e.g., `RecursiveCharacterTextSplitter`) use natural text boundaries like paragraphs, sentences, or sections to create coherent chunks.
- Fixed-size splitters (e.g., `CharacterTextSplitter`) divide text into uniform segments based on character count, which can be useful for ensuring consistent chunk sizes.
    * by token count (more accurate for LLMs)
    * by character count is easier to implement but may not align well with tokenization used by LLMs
- Document-structure-based splitters leverage embeddings to group related content together, preserving context and meaning.
    * Markdown: Split based on headers (e.g., #, ##, ###)
    * HTML: Split using tags
    * JSON: Split by object or array elements
    * Code: Split by functions, classes, or logical blocks

### Text structure-based

Text is naturally organized into hierarchical units such as paragraphs, sentences, and words. We can leverage this inherent structure to inform our splitting strategy, creating split that maintain natural language flow, maintain semantic coherence within split, and adapts to varying levels of text granularity. 

Recommended for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [1]:
!uv pip install -qU langchain-text-splitters

In [4]:
#read and aggreatate all *.md files recursively from root folder
import os
from pathlib import Path
root_dir = Path('../')
md_files = list(root_dir.rglob('*.md'))
content = ""
documents = []
for md_file in md_files:
    with open(md_file, 'r', encoding='utf-8') as f:
        _ = f.read()
        content += _ + "\n\n"
        #metadata = {'source': str(md_file)}
        #documents.append(Document(page_content=_, metadata=metadata))

In [70]:
#RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

_size = 1_000
_overlap = _size // 5  # 20% overlap
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=_size, # size of each chunk, determined based on length_function
    chunk_overlap=_overlap, # overlap between chunks, help to maintain context, mitigate info loss at boundaries where split occurs
    length_function=len, # function to measure length of text, default is len() which counts characters
    separators=["\n\n", "\n", " ", ""], # prioritize splitting at natural boundaries
    is_separator_regex=False,
)
texts = text_splitter.create_documents([content])
print(f"Created {len(texts)} chunks, average {sum(len(t.page_content) for t in texts) / len(texts):.2f} characters each\n")
for i, text in enumerate(texts[:3]):
    print(f"=== Chunk {i} (size: {len(text.page_content)} characters) ===")    
    print(f"{text.page_content[:_size // 4]} \033[91m[...]\033[0m  {text.page_content[-_overlap:]}")
    print()

Created 578 chunks, average 781.74 characters each

=== Chunk 0 (size: 786 characters) ===
# 🤖 ai-crash-course
### 🚀 Your Fast-Track to Becoming an AI Expert!

[![Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE) [![Cours [...]   have to confine itself to methods that are biologically observable."*
> **— John McCarthy, 2007**

---

## 🗺️ Reference Roadmaps

Explore these comprehensive learning paths to deepen your AI journey:

=== Chunk 1 (size: 995 characters) ===
<table>
<tr>
<td align="center" width="25%">
<a href="https://roadmap.sh/ai-engineer">
<img src="https://api.iconify.design/mdi:robot.svg?color=%234285f4" width="48" height="48" alt="AI Engineer"/>
<br/><b>AI Engineer</b>
</a>
</td>
<td align="center [...]  mg src="https://api.iconify.design/mdi:shield-alert.svg?color=%23ea4335" width="48" height="48" alt="AI Red Teaming"/>
<br/><b>AI Red Teaming</b>
</a>
<

### Length-based

An intuitive strategy is to split documents based on their length. This simple yet effective approach ensures that each chunk doesn't exceed a specified size limit. Key benefits of length-based splitting:

* Straightforward implementation
* Consistent chunk sizes
* Easily adaptable to different model requirements

Types of length-based splitting:

* Token-based: Splits text based on the number of tokens, which is useful when working with language models.
* Character-based: Splits text based on the number of characters, which can be more consistent across different types of text.

In [75]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", 
    chunk_size=1_000,  #tokens, but can be LARGER than the chunk size measured by tiktoken
    chunk_overlap=0
)
texts = text_splitter.split_text(content)
print(f"Created {len(texts)} chunks, average {sum(len(t) for t in texts) / len(texts):.2f} characters each\n")
for i, text in enumerate(texts[:3]):
    print(f"=== Chunk {i} (size: {len(text)} characters) ===")    
    print(f"{text[:200]} \033[91m[...]\033[0m  {text[-30:]}")
    print()



Created a chunk of size 1568, which is longer than the specified 1000
Created a chunk of size 1727, which is longer than the specified 1000
Created a chunk of size 1727, which is longer than the specified 1000


Created 112 chunks, average 3684.87 characters each

=== Chunk 0 (size: 1828 characters) ===
# 🤖 ai-crash-course
### 🚀 Your Fast-Track to Becoming an AI Expert!

[![Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [![License](https://img.shields. [...]   intensive AI training program

=== Chunk 1 (size: 2967 characters) ===
| 📅 Week | 📖 Module | 🎓 Learning Goals | 🔑 Key Topics |
|:-------:|-----------|-------------------|---------------|
| [**01**](./01/README.md) | **🔰 AI Engineer Basics** | Understand role of AI Engine [...]  M providers

### 🚀 Quick Start

=== Chunk 2 (size: 3554 characters) ===
```pwsh
# 🐍 uv
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
# linux: curl -LsSf https://astral.sh/uv/install.sh | sh
uv --version

# 1️⃣ Clone the repository
git  [...]  mazon-reviews-sentiment.ipynb)



In [79]:
# ensure that each split is smaller than chunk_size
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(chunk_size=1_000, chunk_overlap=0)

texts = text_splitter.split_text(content)
print(f"Created {len(texts)} chunks, average {sum(len(t) for t in texts) / len(texts):.2f} characters each\n")
for i, text in enumerate(texts[:3]):
    print(f"=== Chunk {i} (size: {len(text)} characters) ===")    
    print(f"{text[:200]} \033[91m[...]\033[0m  {text[-30:]}")
    print()

Created 131 chunks, average 3152.69 characters each

=== Chunk 0 (size: 2743 characters) ===
# 🤖 ai-crash-course
### 🚀 Your Fast-Track to Becoming an AI Expert!

[![Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [![License](https://img.shields. [...]  g |
| [**05**](./05/README.md)

=== Chunk 1 (size: 2781 characters) ===
 | **🧬 Tokenizer & Embeddings** | Learn tokenization, embeddings | • 🎫 Token management, moderation<br/> • ❓ What are embeddings?<br/>• 🎯 Use cases: semantic search, recsys |
| [**06**](./06/README.md [...]  

# 5️⃣ Start learning! 🎉
code

=== Chunk 2 (size: 3502 characters) ===
 .
```

- Note for CUDA users: torch installation &  set `TORCH_CUDA_ARCH_LIST` in `.env` according to your GPU
```sh
nvcc --version
nvidia-smi
#check CUDA capacity, e.g., (12, 1)
python -c "import to [...]  mportant emails as spam)
**Rec



In [10]:
!uv pip install -qU nltk

**NLTK**: The Natural Language Toolkit (NLTK) is a popular Python library for working with human language data. It provides a trained sentence tokenizer that can split the text into sentences, helping to create more meaningful chunks.

<note>Best for English text, several other languages are also supported via `punkt` tokenizer models.</note>

In [ ]:
import nltk
nltk.download(['maxent_ne_chunker_tab', 'words', 'punkt'], quiet=True)
sentence = "The name Artificial Intelligence was coined in 1956 at a conference at Dartmouth College in Hanover."
tokens = nltk.word_tokenize(sentence, language='english')
print(tokens)
# tagged tokens
tagged = nltk.pos_tag(tokens)
print(tagged)
# identify named entities
entities = nltk.chunk.ne_chunk(tagged)
print(entities)

['The', 'name', 'Artificial', 'Intelligence', 'was', 'coined', 'in', '1956', 'at', 'a', 'conference', 'at', 'Dartmouth', 'College', 'in', 'Hanover', '.']
[('The', 'DT'), ('name', 'NN'), ('Artificial', 'NNP'), ('Intelligence', 'NNP'), ('was', 'VBD'), ('coined', 'VBN'), ('in', 'IN'), ('1956', 'CD'), ('at', 'IN'), ('a', 'DT'), ('conference', 'NN'), ('at', 'IN'), ('Dartmouth', 'NNP'), ('College', 'NNP'), ('in', 'IN'), ('Hanover', 'NNP'), ('.', '.')]
(S
  The/DT
  name/NN
  (ORGANIZATION Artificial/NNP Intelligence/NNP)
  was/VBD
  coined/VBN
  in/IN
  1956/CD
  at/IN
  a/DT
  conference/NN
  at/IN
  (ORGANIZATION Dartmouth/NNP College/NNP)
  in/IN
  (GPE Hanover/NNP)
  ./.)


![image.png](./a787ccfe_image.png)

In [7]:
from langchain_text_splitters import NLTKTextSplitter

text_splitter = NLTKTextSplitter(chunk_size=1_000)
texts = text_splitter.split_text(content)
print(f"Created {len(texts)} chunks, average {sum(len(t) for t in texts) / len(texts):.2f} characters each\n")
for i, text in enumerate(texts[:3]):
    print(f"=== Chunk {i} (size: {len(text)} characters) ===")    
    print(f"{text[:200]} \033[91m[...]\033[0m  {text[-30:]}")
    print()

Created a chunk of size 4863, which is longer than the specified 1000
Created a chunk of size 1219, which is longer than the specified 1000
Created a chunk of size 1897, which is longer than the specified 1000
Created a chunk of size 1156, which is longer than the specified 1000
Created a chunk of size 2110, which is longer than the specified 1000
Created a chunk of size 1271, which is longer than the specified 1000
Created a chunk of size 1039, which is longer than the specified 1000
Created a chunk of size 1297, which is longer than the specified 1000
Created a chunk of size 1296, which is longer than the specified 1000
Created a chunk of size 1080, which is longer than the specified 1000
Created a chunk of size 2761, which is longer than the specified 1000
Created a chunk of size 4134, which is longer than the specified 1000
Created a chunk of size 2220, which is longer than the specified 1000
Created a chunk of size 1100, which is longer than the specified 1000
Created a chunk of s

Created 444 chunks, average 1012.50 characters each

=== Chunk 0 (size: 661 characters) ===
# 🤖 ai-crash-course
### 🚀 Your Fast-Track to Becoming an AI Expert!

[!

[Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [!

[License](https://img.shie [...]  t are biologically observable.

=== Chunk 1 (size: 4863 characters) ===
"*
> **— John McCarthy, 2007**

---

## 🗺️ Reference Roadmaps

Explore these comprehensive learning paths to deepen your AI journey:

<table>
<tr>
<td align="center" width="25%">
<a href="https://road [...]  nts.txt

# 5️⃣ Start learning!

=== Chunk 2 (size: 849 characters) ===
🎉
code .

```

- Note for CUDA users: torch installation &  set `TORCH_CUDA_ARCH_LIST` in `.env` according to your GPU
```sh
nvcc --version
nvidia-smi
#check CUDA capacity, e.g., (12, 1)
python -c "im [...]  
=> [week 01](./01/README.md)!



### Document structure-based

Some documents have an inherent structure, such as HTML, Markdown, or JSON files. In these cases, it's beneficial to split the document based on its structure, as it often naturally groups semantically related text. Key benefits of structure-based splitting:

* Preserves the logical organization of the document
* Maintains context within each chunk
* Can be more effective for downstream tasks like retrieval or summarization

Examples of structure-based splitting:

* Markdown: Split based on headers (e.g., #, ##, ###)
> When a full paragraph or document is embedded, the embedding process considers both the overall context and the relationships between the sentences and phrases within the text. This can result in a more comprehensive vector representation that captures the broader meaning and themes of the text.
* HTML: Split using tags
* JSON: Split by object or array elements
* Code: Split by functions, classes, or logical blocks

In [ ]:
# split based on markdown headers
from langchain_text_splitters import MarkdownHeaderTextSplitter
headers_to_split_on = [
    ("#", "title"),
    ("##", "topic"),
    ("###", "detail"),
    #("####", "note"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers = True, # default True, headers are removed from content and stored in metadata
    return_each_line=False # default False, lines are aggregated based on the headers spec
)
md_header_splits = markdown_splitter.split_text(content)
print(md_header_splits[0].metadata)  # metadata contains header info

print(f"Created {len(md_header_splits)} chunks, average {sum(len(t.page_content) for t in md_header_splits) / len(md_header_splits):.2f} characters each\n")
for i, text in enumerate(md_header_splits[:5]):
    print(f"=== Chunk {i} (size: {len(text.page_content)} characters) ===")    
    print(f"Header: {text.metadata.get('title')} > {text.metadata.get('topic')} > {text.metadata.get('detail')} > {text.metadata.get('note')} \n---")
    print(f"{text.page_content}")
    print()

{'title': '🤖 ai-crash-course'}
Created 2056 chunks, average 201.22 characters each

=== Chunk 0 (size: 19 characters) ===
Header: 🤖 ai-crash-course > None > None > None 
---
# 🤖 ai-crash-course

=== Chunk 1 (size: 47 characters) ===
Header: 🤖 ai-crash-course > None > 🚀 Your Fast-Track to Becoming an AI Expert! > None 
---
### 🚀 Your Fast-Track to Becoming an AI Expert!

=== Chunk 2 (size: 273 characters) ===
Header: 🤖 ai-crash-course > None > 🚀 Your Fast-Track to Becoming an AI Expert! > None 
---
[![Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE) [![Course Progress](https://img.shields.io/badge/Lessons-10%20Weeks-orange.svg)](#bootcamp-schedule)

=== Chunk 3 (size: 3 characters) ===
Header: 🤖 ai-crash-course > None > 🚀 Your Fast-Track to Becoming an AI Expert! > None 
---
---

=== Chunk 4 (size: 335 characters) ===
Header: 🤖 ai-crash-course > None > 🚀 Your Fast-Tra

In [ ]:
# combine approaches:
# split based on markdown headers + constrain chunk size
from langchain_text_splitters import MarkdownHeaderTextSplitter
headers_to_split_on = [
    ("#", "title"),
    ("##", "topic"),
    ("###", "detail"),
    #("####", "note"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers = False, # default True, headers are removed from content and stored in metadata
    return_each_line=False # default False, lines are aggregated based on the headers spec
)
md_header_splits = markdown_splitter.split_text(content)
print(md_header_splits[0].metadata)  # metadata contains header info

# constrain chunk size with RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
_size = 10_000
_overlap = 100 #smaller overlap since headers are preserved
text_splitter = RecursiveCharacterTextSplitter(chunk_size=_size, chunk_overlap=_overlap)
texts = text_splitter.split_documents(md_header_splits)
print(f"Created {len(texts)} chunks, average {sum(len(t.page_content) for t in texts) / len(texts):.2f} characters each\n")
for i, text in enumerate(texts[:3]):
    print(f"=== Chunk {i} (size: {len(text.page_content)} characters) ===")    
    print(f"Header: {text.metadata.get('title')} > {text.metadata.get('topic')} > {text.metadata.get('detail')} > {text.metadata.get('note')} \n---")
    print(f"{text.page_content}")
    print()

{'title': '🤖 ai-crash-course', 'detail': '🚀 Your Fast-Track to Becoming an AI Expert!'}
Created 357 chunks, average 1173.63 characters each

=== Chunk 0 (size: 695 characters) ===
Header: 🤖 ai-crash-course > None > 🚀 Your Fast-Track to Becoming an AI Expert! > None 
---
# 🤖 ai-crash-course  
### 🚀 Your Fast-Track to Becoming an AI Expert!  
[![Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE) [![Course Progress](https://img.shields.io/badge/Lessons-10%20Weeks-orange.svg)](#bootcamp-schedule)  
---  
> *"[Artificial intelligence is] the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to the similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that are biologically observable."*
> **— John McCarthy, 2007**  
---

=== Chunk 1 (size: 1071 characte

In [54]:
import json
import requests
from langchain_text_splitters import RecursiveJsonSplitter

# This is a large nested json object and will be loaded as a python dict
json_data = requests.get("https://api.smith.langchain.com/openapi.json").json()
splitter = RecursiveJsonSplitter(max_chunk_size=1_000)

docs = splitter.create_documents(texts=[json_data])
print(f"Created {len(docs)} chunks, average {sum(len(t.page_content) for t in docs) / len(docs):.2f} characters each\n")
for i, text in enumerate(docs[:3]):
    print(f"=== Chunk {i} (size: {len(text.page_content)} characters) ===")    
    print(f"{text.page_content}")
    print()

Created 744 chunks, average 972.82 characters each

=== Chunk 0 (size: 810 characters) ===
{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n", "version": "0.1.0"}, "paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["tracer-sessions"], "summary": "Get Tracing Project Prebuilt Dashboard", "description": "Get a prebuilt dashboard for a tracing project.", "operationId": "get_tracing_project_prebuilt_dashboard_api_v1_sessions__session_id__dashboard_post", "security": [{"API Key": []}, {"Tenant ID": []}, {"Bearer Auth": []}]}}}}

=== Chunk 1 (size: 938 characters) ===
{"paths": {"/api/v1/sessions/{session_id}/dashboard": {"po

In [70]:
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)
print([e.value for e in Language])
RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

# Call the function
hello_world()
"""
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=1_000, chunk_overlap=0
)
python_docs = python_splitter.create_documents([PYTHON_CODE])
print(python_docs)


TS_CODE = """
function helloWorld(): void {
  console.log("Hello, World!");
}

// Call the function
helloWorld();
"""

ts_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.TS, chunk_size=100, chunk_overlap=0
)
ts_docs = ts_splitter.create_documents([TS_CODE])
print(ts_docs)

print("=== markdown ===")
md_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN, chunk_size=1_000, chunk_overlap=0
)
md_docs = md_splitter.create_documents([content])
for i, doc in enumerate(md_docs[:3]):
    print(f"=== Chunk {i} (size: {len(doc.page_content)} characters) ===")
    print(doc)

['cpp', 'go', 'java', 'kotlin', 'js', 'ts', 'php', 'proto', 'python', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex', 'html', 'sol', 'csharp', 'cobol', 'c', 'lua', 'perl', 'haskell', 'elixir', 'powershell', 'visualbasic6']
[Document(metadata={}, page_content='def hello_world():\n    print("Hello, World!")\n\n# Call the function\nhello_world()')]
[Document(metadata={}, page_content='function helloWorld(): void {\n  console.log("Hello, World!");\n}'), Document(metadata={}, page_content='// Call the function\nhelloWorld();')]
=== markdown ===
=== Chunk 0 (size: 689 characters) ===
page_content='# 🤖 ai-crash-course
### 🚀 Your Fast-Track to Becoming an AI Expert!

[![Python](https://img.shields.io/badge/Python-3.12+-blue.svg)](https://www.python.org/downloads/) [![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE) [![Course Progress](https://img.shields.io/badge/Lessons-10%20Weeks-orange.svg)](#bootcamp-schedule)

---

> *"[Artificial intelligence is] the s

In [77]:
#html: analogous to markdown header splitter
from langchain_text_splitters import HTMLHeaderTextSplitter

headers_to_split_on = [
    ("h1", "h1"),
    ("h2", "h2"),
    ("h3", "h3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)
html_header_splits = html_splitter.split_text_from_url("https://plato.stanford.edu/entries/goedel/")
for i, doc in enumerate(html_header_splits[:3]):
    print(f"=== Chunk {i} (size: {len(doc.page_content)} characters) ===")
    print(doc) 

=== Chunk 0 (size: 1013 characters) ===
page_content='End container NOTE: Script required for drop-down button to work (mirrors).  
End header wrapper End content End footer  
End header  
End navigation End search  
Stanford Encyclopedia of Philosophy  
Menu  
Browse  
Table of Contents  
What's New  
Random Entry  
Chronological  
Archives  
About  
Editorial Information  
About the SEP  
Editorial Board  
How to Cite the SEP  
Special Characters  
Advanced Tools  
Contact  
Support SEP  
Support the SEP  
PDFs for SEP Friends  
Make a Donation  
SEPIA for Libraries  
Begin article sidebar End article sidebar NOTE: Article content must have two wrapper divs: id="article" and id="article-content" End article NOTE: article banner is outside of the id="article" div. End article-banner  
Entry Navigation  
Entry Contents  
Bibliography  
Academic Tools  
Friends PDF Preview  
Author and Citation Info  
Back to Top  
End article-content  
BEGIN ARTICLE HTML #aueditable DO NOT MODIFY THIS 